### 살시간 전력 현황
문제: API 동기화 까지 2시간 - 최대 1일 걸림 -> 그래서 현재 데이터 불러올 수 없음

In [ ]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET

KMA_API_KEY = "2CXA78d-TKelwO_HfoynZg"
POWER_API_KEY = "4898e1e9d1fb4330c32a1c4625a551fac391c606c58aab3614339ab86074969b"

url = "https://openapi.kpx.or.kr/openapi/sukub5mToday/getSukub5mToday"

params = {
    "serviceKey": POWER_API_KEY,
    "numOfRows": 1
}

response = requests.get(url, params=params)
content = response.content.decode("utf-8")
content

'<?xml version="1.0" encoding="UTF-8" standalone="yes"?><response><header><resultCode>00</resultCode><resultMsg>OK</resultMsg><pageNo>1</pageNo><numOfRows>1000</numOfRows><totalCount>193</totalCount><pageSize>1</pageSize><startPage>1</startPage></header><body><items><item><baseDatetime>20260702000000</baseDatetime><suppAbility>98755.9</suppAbility><currPwrTot>63905.5</currPwrTot><forecastLoad>80300.0</forecastLoad><suppReservePwr>34850.4</suppReservePwr><suppReserveRate>54.5342</suppReserveRate><operReservePwr>12758.7</operReservePwr><operReserveRate>19.606</operReserveRate></item><item><baseDatetime>20260702000500</baseDatetime><suppAbility>98779.5</suppAbility><currPwrTot>63605.3</currPwrTot><forecastLoad>80300.0</forecastLoad><suppReservePwr>35174.2</suppReservePwr><suppReserveRate>55.3007</suppReserveRate><operReservePwr>12850.0</operReservePwr><operReserveRate>19.8299</operReserveRate></item><item><baseDatetime>20260702001000</baseDatetime><suppAbility>98735.7</suppAbility><currPw

In [12]:

import requests
import pandas as pd
import xml.etree.ElementTree as ET

POWER_API_KEY = "4898e1e9d1fb4330c32a1c4625a551fac391c606c58aab3614339ab86074969b"

url = "https://openapi.kpx.or.kr/openapi/sukub5mToday/getSukub5mToday"

params = {
    "serviceKey": POWER_API_KEY,
    "numOfRows": 1
}

response = requests.get(url, params=params)
# response.raise_for_status()

content = response.content.decode("utf-8")


def kpx_response_to_dataframe(content: str) -> pd.DataFrame:
    root = ET.fromstring(content)

    result_code = root.findtext(".//resultCode")
    result_msg = root.findtext(".//resultMsg")
    if result_code and result_code != "00":
        raise ValueError(f"API error {result_code}: {result_msg or 'Unknown error'}")

    items = root.findall(".//item")
    rows = []

    for item in items:
        row = {
            child.tag: child.text
            for child in item
        }
        rows.append(row)

    df = pd.DataFrame(rows)

    num_cols = [
        "suppAbility",
        "currPwrTot",
        "forecastLoad",
        "suppReservePwr",
        "suppReserveRate",
        "operReservePwr",
        "operReserveRate"
    ]

    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "baseDatetime" in df.columns:
        df["baseDatetime"] = pd.to_datetime(
            df["baseDatetime"],
            format="%Y%m%d%H%M%S",
            errors="coerce"
        )

    return df


df = kpx_response_to_dataframe(content)

df

,baseDatetime,suppAbility,currPwrTot,forecastLoad,suppReservePwr,suppReserveRate,operReservePwr,operReserveRate
0,2026-07-02 00:00:00,98755.9,63905.5,80300.0,34850.4,54.5342,12758.7,19.6060
1,2026-07-02 00:05:00,98779.5,63605.3,80300.0,35174.2,55.3007,12850.0,19.8299
2,2026-07-02 00:10:00,98735.7,63230.0,80300.0,35505.7,56.1533,13062.2,20.2804
3,2026-07-02 00:15:00,98795.0,62827.0,80300.0,35967.9,57.2491,12920.2,20.1818
4,2026-07-02 00:20:00,98832.8,62432.6,80300.0,36400.1,58.3031,12961.3,20.3471
...,...,...,...,...,...,...,...,...
188,2026-07-02 15:40:00,103489.0,74694.5,79900.0,28794.4,38.5496,17134.3,22.2490
189,2026-07-02 15:45:00,103489.0,74741.9,79900.0,28747.2,38.4620,17417.7,22.6263
190,2026-07-02 15:50:00,103399.0,74936.7,79900.0,28462.7,37.9824,17196.9,22.2831
191,2026-07-02 15:55:00,103412.0,75226.8,79900.0,28185.5,37.4673,17131.1,22.0984


In [ ]:

import requests
import pandas as pd
import xml.etree.ElementTree as ET

POWER_API_KEY = "4898e1e9d1fb4330c32a1c4625a551fac391c606c58aab3614339ab86074969b"

url = "https://openapi.kpx.or.kr/openapi/sukub5mToday/getSukub5mToday"

params = {
    "serviceKey": POWER_API_KEY
}

response = requests.get(url, params=params)
# response.raise_for_status()

content = response.content.decode("utf-8")


def kpx_response_to_dataframe(content: str) -> pd.DataFrame:
    root = ET.fromstring(content)

    result_code = root.findtext(".//resultCode")
    result_msg = root.findtext(".//resultMsg")
    if result_code and result_code != "00":
        raise ValueError(f"API error {result_code}: {result_msg or 'Unknown error'}")

    items = root.findall(".//item")
    rows = []

    for item in items:
        row = {
            child.tag: child.text
            for child in item
        }
        rows.append(row)

    df = pd.DataFrame(rows)

    num_cols = [
        "suppAbility",
        "currPwrTot",
        "forecastLoad",
        "suppReservePwr",
        "suppReserveRate",
        "operReservePwr",
    ]

    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "baseDatetime" in df.columns:
        df["baseDatetime"] = pd.to_datetime(
            df["baseDatetime"],
            format="%Y%m%d%H%M%S",
            errors="coerce"
        )

    return df


df = kpx_response_to_dataframe(content)

df

,baseDatetime,suppAbility,currPwrTot,forecastLoad,suppReservePwr,suppReserveRate,operReservePwr,operReserveRate
0,2026-07-02 00:00:00,98755.9,63905.5,80300.0,34850.4,54.5342,12758.7,19.606
1,2026-07-02 00:05:00,98779.5,63605.3,80300.0,35174.2,55.3007,12850.0,19.8299
2,2026-07-02 00:10:00,98735.7,63230.0,80300.0,35505.7,56.1533,13062.2,20.2804
3,2026-07-02 00:15:00,98795.0,62827.0,80300.0,35967.9,57.2491,12920.2,20.1818
4,2026-07-02 00:20:00,98832.8,62432.6,80300.0,36400.1,58.3031,12961.3,20.3471
...,...,...,...,...,...,...,...,...
187,2026-07-02 15:35:00,103445.0,74467.5,79900.0,28977.8,38.9133,17470.5,22.7643
188,2026-07-02 15:40:00,103489.0,74694.5,79900.0,28794.4,38.5496,17134.3,22.249
189,2026-07-02 15:45:00,103489.0,74741.9,79900.0,28747.2,38.4620,17417.7,22.6263
190,2026-07-02 15:50:00,103399.0,74936.7,79900.0,28462.7,37.9824,17196.9,22.2831


In [20]:
import os
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from dotenv import load_dotenv

load_dotenv()

POWER_API_KEY = os.getenv("POWER_API_KEY")

POWER_URL = "https://openapi.kpx.or.kr/openapi/sukub5mToday/getSukub5mToday"


def load_current_power_reserve():
    params = {
        "serviceKey": POWER_API_KEY,
        "numOfRows": 1   # 최신 1건 요청 (API 기본 정렬 기준에 따름)
    }

    response = requests.get(POWER_URL, params=params, timeout=10)
    response.raise_for_status()

    root = ET.fromstring(response.content)

    result_code = root.findtext(".//resultCode")
    result_msg = root.findtext(".//resultMsg")

    if result_code and result_code != "00":
        raise ValueError(f"API error {result_code}: {result_msg}")

    items = root.findall(".//item")

    if not items:
        raise ValueError("전력거래소 응답에 데이터가 없습니다.")

    rows = []

    for item in items:
        rows.append({
            child.tag: child.text
            for child in item
        })

    df = pd.DataFrame(rows)

    df["suppReserveRate"] = pd.to_numeric(
        df["suppReserveRate"],
        errors="coerce"
    )

    df["baseDatetime"] = pd.to_datetime(
        df["baseDatetime"],
        format="%Y%m%d%H%M%S",
        errors="coerce"
    )

    # 혹시 여러 건이 오더라도 최신 데이터 사용
    df = df.sort_values("baseDatetime", ascending=False)

    row = df.iloc[0]

    return {
        "baseDatetime": None if pd.isna(row["baseDatetime"]) else row["baseDatetime"].strftime("%Y-%m-%d %H:%M:%S"),
        "suppReserveRate": None if pd.isna(row["suppReserveRate"]) else float(row["suppReserveRate"]),
    }

In [22]:
load_current_power_reserve()

{'baseDatetime': '2026-07-02 16:45:00', 'suppReserveRate': 34.06}